# Supplementary Carry-On Shopping Answer Analysis  
## Comparison 01, baseline-based multi-prompt version — FIXED  
### Supplementary Check: 2023/2024 Original Baseline vs One-Product 2026 Current Replacement

This notebook analyzes:

**2023/2024 original baseline vs 2026 current one-product replacement**

Purpose: check how much the actual product page changed over time.  
Interpretation: general website/text change.

## Fixed in this version

This version is stricter about multi-prompt matching. It will not silently summarize only Q1–5 if your current files contain Q6–20 but the matching all-original baseline answer files are missing or not detected.

Main fixes:

- Searches baseline files with `*.txt*`, not only `*.txt`.
- Accepts `2023vs2026`, `2023_2024vs2026`, and similar current-file naming variants.
- Handles product filename typos such as `desley` and `inusea`.
- Ignores GEO files in Comparison 01 by default.
- Matches by `product + question_id`.
- Adds coverage diagnostics for baseline question IDs, current question IDs, matched IDs, and missing baseline IDs.
- Raises a clear error when current questions are missing matching baseline questions, unless you set `ALLOW_PARTIAL_MATCH = True`.

Important: to get Q1–20 results, the baseline folder must contain all-original baseline answer files covering Q1–20. For example, baseline files may be named `baseline_1.txt`, `baseline_6.txt`, `baseline_11.txt`, and `baseline_16.txt`, or `baseline_1.text`, `baseline_6.text`, `baseline_11.text`, and `baseline_16.text`, or any other names, as long as they contain the correct `Question 1:` ... `Question 20:` blocks.


Additional update in this version: baseline discovery accepts both `.txt` and `.text` files, including names such as `baseline_1.text`, `baseline_6.text`, `baseline_11.text`, and `baseline_16.text`.

## Expected folder structure

The notebook searches flexibly, but the main expected structure is:

```text
code/
└── data/
    └── results/
        └── shopping_answers/
            ├── baseline/
            │   ├── baseline.txt
            │   ├── baseline_1.txt or baseline_1.text        # optional
            │   ├── baseline_6.txt or baseline_6.text        # optional
            │   ├── baseline_11.txt or baseline_11.text      # optional
            │   └── baseline_16.txt or baseline_16.text      # optional
            └── 01_2023_2024_original_vs_2026_current/
                ├── beis_2023vs2026_1.txt
                ├── beis_2023vs2026_6.txt
                ├── beis_2023vs2026_16.txt
                ├── delsey_2023vs2026_1.txt
                ├── delsey_2023vs2026_6.txt
                ├── desley_2023vs2026_16.txt
                └── ...
```

Optional product-page text folders used for PAIS-style text similarity:

```text
code/data/filtered/old/
code/data/filtered/current/
```


## CSV outputs

The notebook saves CSV files to:

```text
code/data/results/tables/01_2023_2024_original_vs_2026_current_baseline_based/
```

Saved files:

```text
file_manifest_comparison01.csv
baseline_parsed_answers.csv
current_parsed_answers_by_file.csv
baseline_question_product_metrics.csv
current_question_product_metrics.csv
matched_current_baseline_question_product_delta.csv
baseline_product_summary_matched.csv
current_product_summary_matched.csv
current_vs_baseline_delta_by_product.csv
overall_current_vs_baseline_summary.csv
product_text_load_status.csv
```


In [1]:
from pathlib import Path
import re
import json
import math
import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 300)


In [2]:
# =============================
# Configuration
# =============================

PRODUCTS = ["Away", "BR", "Century", "July", "Rimowa"]
N_PRODUCTS = len(PRODUCTS)

COMPARISON_NAME = "01_2023_2024_original_vs_2026_current"
OUTPUT_NAME = "01_2023_2024_original_vs_2026_current_baseline_based"

# Optional manual overrides.
# If automatic path detection fails, set these manually.
BASELINE_DIR = None
BASELINE_FILES = None     # Example: [Path(".../baseline_1.txt"), Path(".../baseline_6.text")]
COMPARISON_DIR = None

# Optional manual overrides for product-page text folders.
# These are used only for PAIS-style product-text-to-answer similarity.
OLD_TEXT_DIR = None
CURRENT_TEXT_DIR = None

# For Comparison 01, GEO files should not be analyzed even if they are in the same folder.
# If your Q11-15 files are actually current-vs-2026 runs but accidentally contain "GEO" in the filename,
# rename them to something like beis_2023vs2026_11.txt instead of changing this flag.
IGNORE_GEO_FILES_IN_COMPARISON01 = True

# IMPORTANT FIX:
# If False, the notebook raises an error when some current question IDs do not have matching baseline question IDs.
# This prevents accidentally reporting only Q1-5 when Q6-20 current files exist but baseline files are missing.
ALLOW_PARTIAL_MATCH = False

# Expected final benchmark questions. Used for diagnostics only.
EXPECTED_QUESTION_IDS = list(range(1, 21))

# If you want to restrict analysis to specific question IDs, set a list like:
# QUESTION_ID_FILTER = list(range(1, 21))
QUESTION_ID_FILTER = None


In [3]:
# =============================
# Robust path detection
# =============================

def get_candidate_roots():
    cwd = Path.cwd().resolve()
    roots = []
    for p in [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
        cwd / "code",
        cwd.parent / "code",
        cwd.parent.parent / "code",
        cwd.parent.parent.parent / "code",
    ]:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if p.exists() and rp not in roots:
            roots.append(rp)
    return roots

def find_dir(rel_candidates):
    for root in get_candidate_roots():
        for rel in rel_candidates:
            candidate = root / rel
            if candidate.exists() and candidate.is_dir():
                return candidate.resolve()
    return None

def normalize_filename(name: str) -> str:
    return name.lower().replace("é", "e").replace(" ", "").replace("-", "_")

PRODUCT_ALIASES = {
    "Away": ["away"],
    "BR": ["br", "briggs", "riley", "briggsriley", "briggsandriley"],
    "Century": ["century", "hartmann"],
    "July": ["july"],
    "Rimowa": ["rimowa"],
}

def infer_product_from_filename(filename: str) -> str:
    name = normalize_filename(filename)
    for product, aliases in PRODUCT_ALIASES.items():
        if any(alias in name for alias in aliases):
            return product
    return "Unknown"

def is_geo_filename(filename: str) -> bool:
    name = normalize_filename(filename)
    compact = re.sub(r"[^a-z0-9]", "", name)
    return "geo" in compact or "rewrite" in compact

def is_text_like(path: Path) -> bool:
    """Accept normal text answer files.

    Supports both .txt and .text because some prompt outputs may be saved as
    baseline_1.text, baseline_6.text, etc. Also accepts double extensions that
    Windows/Notepad sometimes creates, such as .txt.txt or .text.txt.
    """
    n = path.name.lower()
    return (
        n.endswith(".txt")
        or n.endswith(".txt.txt")
        or n.endswith(".text")
        or n.endswith(".text.txt")
    )

def is_current_replacement_filename(filename: str) -> bool:
    """Accept one-product current replacement answer files for Comparison 01.

    The function scans only the dedicated Comparison 01 folder, so it accepts
    product-known text files even if the filename is simply away_1.txt rather
    than away_2023vs2026_1.txt. GEO/rewrite files are still excluded.
    """
    name = normalize_filename(filename)
    if filename.startswith("."):
        return False
    if not is_text_like(Path(filename)):
        return False
    if IGNORE_GEO_FILES_IN_COMPARISON01 and is_geo_filename(name):
        return False
    if infer_product_from_filename(name) == "Unknown":
        return False
    return True

def infer_prompt_set_from_question_ids(question_ids):
    if not question_ids:
        return np.nan, np.nan, ""
    qmin, qmax = int(min(question_ids)), int(max(question_ids))
    return qmin, qmax, f"{qmin}-{qmax}"

def sort_prompt_set_labels(labels):
    """Sort labels like 1-5, 6-10, 11-15, 16-20 in numeric order."""
    clean = [str(x) for x in labels if str(x).strip() and str(x).lower() != "nan"]
    def key(label):
        m = re.search(r"(\d+)", label)
        return int(m.group(1)) if m else 10**9
    return sorted(set(clean), key=key)

def join_prompt_set_labels(labels):
    return ";".join(sort_prompt_set_labels(labels))

def discover_baseline_files(baseline_dir: Path):
    """Find all all-original baseline answer files.

    Uses both rglob('*.txt*') and rglob('*.text*') so files like baseline_16.txt.txt or baseline_16.text are included.
    Excludes obvious comparison / GEO / current replacement files.
    """
    if baseline_dir is None:
        return []
    # Accept both .txt and .text baseline files.
    # Examples: baseline.txt, baseline_1.txt, baseline_6.text, baseline_16.text.txt.
    candidates = []
    for pattern in ["*.txt*", "*.text*"]:
        candidates.extend([p for p in baseline_dir.rglob(pattern) if p.is_file() and is_text_like(p)])
    files = sorted(set(candidates))
    valid = []
    for p in files:
        n = normalize_filename(p.name)
        compact = re.sub(r"[^a-z0-9]", "", n)
        # Exclude counterfactual/comparison outputs if accidentally copied into the baseline folder.
        banned_tokens = ["vs2026", "vsgeo", "2023vs2026", "20232024vs2026", "20232024vsgeo", "geo_rewrite", "georewrite"]
        if any(tok in compact for tok in banned_tokens):
            continue
        valid.append(p)
    return valid

if BASELINE_DIR is None:
    BASELINE_DIR = find_dir([
        Path("data/results/shopping_answers/baseline"),
        Path("code/data/results/shopping_answers/baseline"),
        Path("results/shopping_answers/baseline"),
        Path("data/results/shopping_answers/00_baseline"),
        Path("code/data/results/shopping_answers/00_baseline"),
        Path("results/shopping_answers/00_baseline"),
        Path("data/results/shopping_answers/00_all_original_2023_2024"),
        Path("code/data/results/shopping_answers/00_all_original_2023_2024"),
        Path("results/shopping_answers/00_all_original_2023_2024"),
    ])
else:
    BASELINE_DIR = Path(BASELINE_DIR).resolve()

if COMPARISON_DIR is None:
    COMPARISON_DIR = find_dir([
        Path("data/results/shopping_answers") / COMPARISON_NAME,
        Path("code/data/results/shopping_answers") / COMPARISON_NAME,
        Path("results/shopping_answers") / COMPARISON_NAME,
    ])
else:
    COMPARISON_DIR = Path(COMPARISON_DIR).resolve()

if OLD_TEXT_DIR is None:
    OLD_TEXT_DIR = find_dir([
        Path("data/filtered/old"),
        Path("code/data/filtered/old"),
        Path("data/results/filtered/old"),
        Path("code/data/results/filtered/old"),
    ])
else:
    OLD_TEXT_DIR = Path(OLD_TEXT_DIR).resolve()

if CURRENT_TEXT_DIR is None:
    CURRENT_TEXT_DIR = find_dir([
        Path("data/filtered/current"),
        Path("code/data/filtered/current"),
        Path("data/results/filtered/current"),
        Path("code/data/results/filtered/current"),
    ])
else:
    CURRENT_TEXT_DIR = Path(CURRENT_TEXT_DIR).resolve()

if BASELINE_FILES is None:
    BASELINE_FILES = discover_baseline_files(BASELINE_DIR)
else:
    BASELINE_FILES = [Path(p).resolve() for p in BASELINE_FILES]

if COMPARISON_DIR is not None:
    candidates = []
    for pattern in ["*.txt*", "*.text*"]:
        candidates.extend([p for p in COMPARISON_DIR.glob(pattern) if p.is_file() and is_text_like(p)])
    all_comparison_txt_files = sorted(set(candidates))
else:
    all_comparison_txt_files = []

TXT_FILES = [p for p in all_comparison_txt_files if is_current_replacement_filename(p.name)]
IGNORED_TXT_FILES = [p for p in all_comparison_txt_files if p not in TXT_FILES]

print("Current working directory:", Path.cwd().resolve())
print("Detected baseline directory:", BASELINE_DIR)
print("Detected baseline files:")
for p in BASELINE_FILES:
    print("  -", p.name)
print("Detected comparison directory:", COMPARISON_DIR)
print("Detected old text directory:", OLD_TEXT_DIR)
print("Detected current text directory:", CURRENT_TEXT_DIR)

if not BASELINE_FILES:
    raise FileNotFoundError(
        "Could not find baseline .txt/.text files. "
        "Please set BASELINE_DIR or BASELINE_FILES manually in the configuration cell. "
        "For Q1-20 analysis, baseline files must cover Q1-20."
    )

if COMPARISON_DIR is None:
    raise FileNotFoundError(
        "Could not find the comparison folder. "
        "Please set COMPARISON_DIR manually in the configuration cell."
    )

print(f"\nIncluded {len(TXT_FILES)} one-product-current txt files:")
for p in TXT_FILES:
    print("  -", p.name, "=>", infer_product_from_filename(p.name))

print(f"\nIgnored {len(IGNORED_TXT_FILES)} txt-like files:")
for p in IGNORED_TXT_FILES:
    reason = "GEO file" if is_geo_filename(p.name) else "not recognized as current replacement"
    print("  -", p.name, f"({reason})")

if not TXT_FILES:
    raise FileNotFoundError(f"No current-replacement .txt/.text files found in {COMPARISON_DIR}")


Current working directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\code
Detected baseline directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline
Detected baseline files:
  - baseline_1.txt
  - baseline_11.txt
  - baseline_16.txt
  - baseline_6.txt
Detected comparison directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current
Detected old text directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\old
Detected current text directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current

Included 20 one-product-current txt files:
  - away_1.txt => Away
  - away_11.txt => Away
  - away_16.txt => Away
  - away_6.txt => Away
  - br_1.txt => BR
  - br_11.txt => BR
  - br_16

In [4]:
# =============================
# Output directory
# =============================

# Based on your structure, if COMPARISON_DIR is:
# code/data/results/shopping_answers/01_...
# then tables go to:
# code/data/results/tables/01_..._baseline_based
RESULTS_DIR = COMPARISON_DIR.parents[1]  # .../results
OUTPUT_DIR = RESULTS_DIR / "tables" / OUTPUT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

file_manifest_rows = []
for p in BASELINE_FILES:
    file_manifest_rows.append({
        "file_role": "baseline_included",
        "path": str(p),
        "filename": p.name,
        "detected_product": "",
        "included": True,
        "reason": "baseline answer file",
    })
for p in TXT_FILES:
    file_manifest_rows.append({
        "file_role": "comparison01_current_included",
        "path": str(p),
        "filename": p.name,
        "detected_product": infer_product_from_filename(p.name),
        "included": True,
        "reason": "current replacement file",
    })
for p in IGNORED_TXT_FILES:
    file_manifest_rows.append({
        "file_role": "comparison01_ignored",
        "path": str(p),
        "filename": p.name,
        "detected_product": infer_product_from_filename(p.name),
        "included": False,
        "reason": "GEO file ignored" if is_geo_filename(p.name) else "not recognized as current replacement",
    })

file_manifest_comparison01 = pd.DataFrame(file_manifest_rows)

print("Results directory:", RESULTS_DIR)
print("CSV output directory:", OUTPUT_DIR)
display(file_manifest_comparison01)


Results directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results
CSV output directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\tables\01_2023_2024_original_vs_2026_current_baseline_based


,file_role,path,filename,detected_product,included,reason
0,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_1.txt,,True,baseline answer file
1,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_11.txt,baseline_11.txt,,True,baseline answer file
2,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_16.txt,baseline_16.txt,,True,baseline answer file
3,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_6.txt,baseline_6.txt,,True,baseline answer file
4,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,away_1.txt,Away,True,current replacement file
5,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_11.txt,away_11.txt,Away,True,current replacement file
6,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_16.txt,away_16.txt,Away,True,current replacement file
7,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_6.txt,away_6.txt,Away,True,current replacement file
8,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\br_1.txt,br_1.txt,BR,True,current replacement file
9,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\br_11.txt,br_11.txt,BR,True,current replacement file


## Parse answer files

The parser expects each answer file to contain question blocks such as:

```text
Question 1:
Answer:
...

Question 2:
Answer:
...

Overall shopper takeaway:
...
```

This version accepts any question ID range, so it can parse prompt sets 1–5, 6–10, 11–15, and 16–20.


In [5]:
# =============================
# Parsing functions
# =============================

QUESTION_BLOCK_RE = re.compile(
    r"Question\s*(\d+)\s*:\s*(.*?)(?=\n\s*Question\s*\d+\s*:|\n\s*Overall shopper takeaway\s*:|\Z)",
    flags=re.IGNORECASE | re.DOTALL,
)

OVERALL_RE = re.compile(
    r"Overall shopper takeaway\s*:\s*(.*)\Z",
    flags=re.IGNORECASE | re.DOTALL,
)

def clean_answer_block(block: str) -> str:
    block = block.strip()
    block = re.sub(r"^\s*Answer\s*:\s*", "", block, flags=re.IGNORECASE).strip()
    block = re.sub(r"\n{3,}", "\n\n", block)
    return block

def parse_answer_file(path: Path, run_type: str, focal_current_product: str = None) -> pd.DataFrame:
    text = path.read_text(encoding="utf-8-sig", errors="replace").replace("\r\n", "\n").replace("\r", "\n")
    rows = []
    qids_found = []

    for m in QUESTION_BLOCK_RE.finditer(text):
        qid = int(m.group(1))
        if QUESTION_ID_FILTER is not None and qid not in set(QUESTION_ID_FILTER):
            continue
        answer = clean_answer_block(m.group(2))
        qids_found.append(qid)
        rows.append({
            "source_file": path.name,
            "source_path": str(path),
            "run_type": run_type,
            "focal_current_product": focal_current_product,
            "comparison": COMPARISON_NAME,
            "question_id": qid,
            "answer_text": answer,
        })

    overall = ""
    om = OVERALL_RE.search(text)
    if om:
        overall = om.group(1).strip()

    qmin, qmax, qlabel = infer_prompt_set_from_question_ids(qids_found)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["prompt_set_start"] = qmin
        df["prompt_set_end"] = qmax
        df["prompt_set_label"] = qlabel
        df["overall_shopper_takeaway"] = overall
    return df

baseline_parts = [
    parse_answer_file(p, run_type="baseline_all_original", focal_current_product=None)
    for p in BASELINE_FILES
]
baseline_parts = [df for df in baseline_parts if not df.empty]
baseline_answers = pd.concat(baseline_parts, ignore_index=True) if baseline_parts else pd.DataFrame()

current_parts = [
    parse_answer_file(
        p,
        run_type="one_product_current",
        focal_current_product=infer_product_from_filename(p.name),
    )
    for p in TXT_FILES
]
current_parts = [df for df in current_parts if not df.empty]
current_answers = pd.concat(current_parts, ignore_index=True) if current_parts else pd.DataFrame()

if baseline_answers.empty:
    raise ValueError("No question blocks were parsed from baseline files.")

if current_answers.empty:
    raise ValueError("No question blocks were parsed from current comparison files.")

# Remove exact duplicate parsed rows if the same file was accidentally included twice.
dedup_cols = ["source_file", "run_type", "focal_current_product", "question_id", "answer_text"]
baseline_answers = baseline_answers.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
current_answers = current_answers.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

baseline_qids = sorted(map(int, baseline_answers["question_id"].unique()))
current_qids = sorted(map(int, current_answers["question_id"].unique()))

print("Baseline parsed rows:", len(baseline_answers))
display(baseline_answers[["source_file", "prompt_set_label", "question_id", "answer_text"]].head(30))

print("Current counterfactual parsed rows:", len(current_answers))
display(current_answers[["source_file", "focal_current_product", "prompt_set_label", "question_id", "answer_text"]].head(30))

print("Focal current products detected by file:")
display(
    current_answers[["source_file", "focal_current_product", "prompt_set_label"]]
    .drop_duplicates()
    .sort_values(["focal_current_product", "prompt_set_label", "source_file"])
)

coverage_by_file = pd.concat([
    baseline_answers.groupby(["run_type", "source_file"], as_index=False).agg(
        focal_current_product=("focal_current_product", lambda x: ""),
        question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(map(int, x)))))),
        n_questions=("question_id", "nunique"),
        prompt_set_label=("prompt_set_label", lambda x: ";".join(sorted(set(map(str, x)))))
    ),
    current_answers.groupby(["run_type", "source_file", "focal_current_product"], as_index=False).agg(
        question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(map(int, x)))))),
        n_questions=("question_id", "nunique"),
        prompt_set_label=("prompt_set_label", lambda x: ";".join(sorted(set(map(str, x)))))
    )
], ignore_index=True, sort=False)

print("Question coverage by parsed file:")
display(coverage_by_file.sort_values(["run_type", "focal_current_product", "source_file"]))

question_coverage_summary = pd.DataFrame([{
    "baseline_question_ids": ",".join(map(str, baseline_qids)),
    "current_question_ids": ",".join(map(str, current_qids)),
    "expected_question_ids": ",".join(map(str, EXPECTED_QUESTION_IDS)),
    "missing_from_baseline_vs_expected": ",".join(map(str, sorted(set(EXPECTED_QUESTION_IDS) - set(baseline_qids)))),
    "missing_from_current_vs_expected": ",".join(map(str, sorted(set(EXPECTED_QUESTION_IDS) - set(current_qids)))),
}])

print("Overall question coverage summary:")
display(question_coverage_summary)


Baseline parsed rows: 20


,source_file,prompt_set_label,question_id,answer_text
0,baseline_1.txt,1-5,1,"I would recommend Away The Carry-On for a 3-5 day trip because its page explicitly labels it for 3-5 days, says it fits 5-7 outfits, and lists a 39.8 L capacity. [Away] It also..."
1,baseline_1.txt,1-5,2,"July Carry On Original seems best if durability and protecting belongings are the top priorities because its page describes an aerospace-grade German polycarbonate shell, reinf..."
2,baseline_1.txt,1-5,3,RIMOWA Essential Lite Cabin seems easiest to move through airports and crowded spaces because it is the lightest listed option at 4.9 lbs and has a Multiwheel system described ...
3,baseline_1.txt,1-5,4,"Away The Carry-On seems best for all-around packing organization because its page describes one compartment for clothes and one for bulkier items such as shoes and toiletries, ..."
4,baseline_1.txt,1-5,5,"July gives the best overall purchase confidence when weighing size, security, warranty or trial, and travel limitations because its page lists 21.5 x 15 x 8.5 inch external dim..."
5,baseline_11.txt,11-15,11,"The safest options on the page text are Away and July, with caveats. Away gives the clearest airline-fit language: it says the bag is its smallest suitcase, ""sized to fit in al..."
6,baseline_11.txt,11-15,12,"July provides the clearest cleaning-related claims: it lists a water-resistant and stain-proof nylon lining, a nylon stain-proof laundry bag, and says all lining is stain-resis..."
7,baseline_11.txt,11-15,13,Away has the strongest concrete sustainability-related evidence because its page says Classic suitcases ship in exterior packaging that is 100% recyclable and the drawstring la...
8,baseline_11.txt,11-15,14,"For travel-day convenience overall, Briggs & Riley seems the most useful because it combines a SmartLink strap for carrying bags together, a PowerPocket for battery pack and ph..."
9,baseline_11.txt,11-15,15,"Away has one of the strongest information pages because it gives clear airline-fit claims, exterior and interior measurements, weight, capacity, materials, included features, r..."


Current counterfactual parsed rows: 100


,source_file,focal_current_product,prompt_set_label,question_id,answer_text
0,away_1.txt,Away,1-5,1,"I would recommend Away The Carry-On first for a 3-5 day trip because its page explicitly says it is perfect for 3-5 day trips, packs 5-7 outfits, fits overhead on most major ai..."
1,away_1.txt,Away,1-5,2,"For durability and protecting belongings, I would recommend July if you prefer a hard-shell suitcase because it uses an aerospace-grade German polycarbonate shell, reinforced b..."
2,away_1.txt,Away,1-5,3,"RIMOWA seems easiest for airports and crowded spaces because it is the lightest option with a stated weight of 4.9 lbs, and its Multiwheel system is described as providing stab..."
3,away_1.txt,Away,1-5,4,"Briggs & Riley seems best for packing organization because it combines one-touch CX expansion and compression, a flat packing surface from the exterior-mounted Outsider handle,..."
4,away_1.txt,Away,1-5,5,"Away gives the best overall purchase confidence for most shoppers because it combines a clearly stated carry-on size of 21.7 x 14.4 x 9 inches, a TSA-approved combination lock,..."
5,away_11.txt,Away,11-15,11,"The safest options for stricter carry-on rules appear to be Away and July, because they give the strongest airline-fit language and full exterior measurements. Away is the smal..."
6,away_11.txt,Away,11-15,12,"July gives the clearest cleaning and appearance-related claims because it mentions water-resistant and stain-proof nylon lining, a nylon stain-proof laundry bag, and stain-resi..."
7,away_11.txt,Away,11-15,13,"Away provides the most concrete sustainability-related information, but the evidence is limited to smaller components and packaging rather than the main suitcase shell. Away sa..."
8,away_11.txt,Away,11-15,14,"Briggs & Riley seems most useful overall for travel-day convenience because it combines practical organization, quick access, and transport features. It has CX expansion and co..."
9,away_11.txt,Away,11-15,15,"Away and July have the strongest information quality for a shopper comparing carry-ons because they give dimensions, weight, capacity, airline-fit language, materials, convenie..."


Focal current products detected by file:


,source_file,focal_current_product,prompt_set_label
0,away_1.txt,Away,1-5
5,away_11.txt,Away,11-15
10,away_16.txt,Away,16-20
15,away_6.txt,Away,6-10
20,br_1.txt,BR,1-5
25,br_11.txt,BR,11-15
30,br_16.txt,BR,16-20
35,br_6.txt,BR,6-10
40,century_1.txt,Century,1-5
45,century_11.txt,Century,11-15


Question coverage by parsed file:


,run_type,source_file,focal_current_product,question_ids,n_questions,prompt_set_label
0,baseline_all_original,baseline_1.txt,,"1,2,3,4,5",5,1-5
1,baseline_all_original,baseline_11.txt,,"11,12,13,14,15",5,11-15
2,baseline_all_original,baseline_16.txt,,"16,17,18,19,20",5,16-20
3,baseline_all_original,baseline_6.txt,,"6,7,8,9,10",5,6-10
4,one_product_current,away_1.txt,Away,"1,2,3,4,5",5,1-5
5,one_product_current,away_11.txt,Away,"11,12,13,14,15",5,11-15
6,one_product_current,away_16.txt,Away,"16,17,18,19,20",5,16-20
7,one_product_current,away_6.txt,Away,"6,7,8,9,10",5,6-10
8,one_product_current,br_1.txt,BR,"1,2,3,4,5",5,1-5
9,one_product_current,br_11.txt,BR,"11,12,13,14,15",5,11-15


Overall question coverage summary:


,baseline_question_ids,current_question_ids,expected_question_ids,missing_from_baseline_vs_expected,missing_from_current_vs_expected
0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",,


### Baseline file naming note

For Q1–20, make sure the baseline folder contains all-original answers covering the same question IDs as the current replacement files. The notebook now accepts either `.txt` or `.text` baseline files, for example:

```text
baseline_1.text    # Q1–5
baseline_6.text    # Q6–10
baseline_11.text   # Q11–15
baseline_16.text   # Q16–20
```

The exact file names do not matter as long as the files contain `Question 1:`, `Question 6:`, `Question 11:`, and `Question 16:` blocks respectively.

## Load product-page texts

The PAIS-style score uses product-page text similarity to cited answer sentences.

If the text files are not found, the notebook still computes citation/rank/feature metrics, while the TF-IDF and n-gram parts of PAIS are set to zero.


In [6]:
# =============================
# Product text loading
# =============================

PRODUCT_FILE_STEMS = {
    "Away": "away.txt",
    "BR": "br.txt",
    "Century": "century.txt",
    "July": "july.txt",
    "Rimowa": "rimowa.txt",
}

def read_text_if_exists(path):
    if path is not None and path.exists():
        return path.read_text(encoding="utf-8-sig", errors="replace")
    return ""

product_texts = {}
load_rows = []

for version_status, folder in [
    ("original_2023_2024", OLD_TEXT_DIR),
    ("current_2026", CURRENT_TEXT_DIR),
]:
    for product, filename in PRODUCT_FILE_STEMS.items():
        path = folder / filename if folder is not None else None
        text = read_text_if_exists(path)
        product_texts[(version_status, product)] = text
        load_rows.append({
            "version_status": version_status,
            "product": product,
            "path": str(path) if path is not None else "",
            "loaded": bool(text.strip()),
            "characters": len(text),
        })

product_text_load_status = pd.DataFrame(load_rows)
display(product_text_load_status)


,version_status,product,path,loaded,characters
0,original_2023_2024,Away,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\old\away.txt,True,4184
1,original_2023_2024,BR,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\old\br.txt,True,4440
2,original_2023_2024,Century,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\old\century.txt,True,1472
3,original_2023_2024,July,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\old\july.txt,True,3403
4,original_2023_2024,Rimowa,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\old\rimowa.txt,True,1686
5,current_2026,Away,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\away.txt,True,6702
6,current_2026,BR,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\br.txt,True,4191
7,current_2026,Century,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\century.txt,True,1987
8,current_2026,July,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\july.txt,True,3029
9,current_2026,Rimowa,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\rimowa.txt,True,1727


## Citation, rank, feature, and PAIS-style metrics


In [7]:
# =============================
# Citation and feature extraction
# =============================

PRODUCT_PATTERN = re.compile(r"\[(Away|BR|B&R|Briggs\s*&\s*Riley|Briggs and Riley|Briggs|Century|Hartmann Century|Hartmann|July|Rimowa|RIMOWA)\]", flags=re.IGNORECASE)

CANONICAL = {
    "away": "Away",
    "br": "BR",
    "b&r": "BR",
    "briggs": "BR",
    "briggs & riley": "BR",
    "briggs and riley": "BR",
    "century": "Century",
    "hartmann": "Century",
    "hartmann century": "Century",
    "july": "July",
    "rimowa": "Rimowa",
}

# Feature categories are intentionally broad and product-page-oriented.
# The added categories help later prompt sets 11-20.
FEATURE_PATTERNS = {
    "trip_length": r"\b(3-5|3–5|4-6|4–6|5-7|5–7|2-5|2–5|day trip|trip length|overnight|weekend|getaway|outfits?)\b",
    "overhead_bin_or_airline_fit": r"\b(overhead|carry-on|carry on|cabin|sizer|airline|domestic airlines?|international|flight|bin|gate-check|gate check|restrictions?|approved)\b",
    "dimensions": r"\b(dimensions?|measurements?|inches|inch|in\.|cm|height|width|depth|linear|exterior|interior)\b",
    "weight": r"\b(weight|lbs?|pounds?|kg|lightweight|lighter|lite|lift)\b",
    "capacity_volume": r"\b(capacity|volume|liters?|litres?|L\b|packing space|room|extra room|capacity-to-weight)\b",
    "material_shell": r"\b(polycarbonate|german polycarbonate|PC/ABS|shell|hard shell|hardside|hard-sided|ballistic nylon|nylon|softside|fabric|aerospace-grade|recycled materials?)\b",
    "durability_protection": r"\b(durable|durability|protect|protection|impact|cracking|corner|guard|bumper|armor|scuff|scratch|resistant|resilience|shock|rough handling|drop|tested|YKK|self-repairing)\b",
    "wheels_mobility": r"\b(wheels?|spinner|rolling|roll|glide|gliding|maneuver|manoeuvre|mobility|PrecisionGlide|MagnaTrac|Hinomoto|SilentMove|Multiwheel|360|crowded|streets?)\b",
    "handle": r"\b(handle|telescopic|trolley|PowerScope|Contour Grip|Outsider|T-Bar|grip|cushioned|soft-grip|GEL|ergonomic)\b",
    "lock_security": r"\b(TSA|lock|security|secure|Travel Sentry|combination|SECURITECH|zip|zipper|intrusion)\b",
    "expansion": r"\b(expand|expandable|expansion|CX|compression and expansion|2-inch|2 inch|1.5|extra space|overpacker|overpackers)\b",
    "packing_organization": r"\b(compartment|compression|pocket|divider|strap|laundry|garment|sleeve|shoe|mesh|organize|organization|clamshell|butterfly|valuables|pouch|small items?)\b",
    "warranty_trial_return": r"\b(warranty|guarantee|trial|return|returns|lifetime|LifetimeCare|10-year|100-day|promise|coverage|reassurance|purchase confidence)\b",
    "price_value": r"\b(price|priced|cost|lower-priced|budget|value|sale|discount|investment|premium)\b",
    "electronics_convenience": r"\b(USB|USB-A|USB-C|charging|charger|battery|power bank|phone|laptop|electronics?|device|tracking|tracker|tech-heavy)\b",
    "cleaning_care": r"\b(clean|cleaning|wipe|wipe-clean|stain|water-resistant|water resistant|scratch|scuff|finish|marks?|dust bag|care)\b",
    "sustainability": r"\b(sustainable|sustainability|responsible|recycled|B Corporation|B Corp|eco-friendly|eco|materials?)\b",
    "comparison_ready_evidence": r"\b(specs?|specifications?|details?|feature details?|trust signals?|buyer caveats?|compare|comparison|clear information|information quality)\b",
    "tradeoff_or_limitation": r"\b(limitation|limit|trade-off|tradeoff|may not|does not|caution|check|vary|not provide|less detail|less complete|harder to compare|however|but|weaker choice)\b",
}

def canonical_product(label: str) -> str:
    return CANONICAL.get(label.lower(), label)

def citation_sequence(text: str):
    return [canonical_product(m.group(1)) for m in PRODUCT_PATTERN.finditer(text or "")]

# Provider-style mention metrics. These parse natural brand/product mentions after
# removing controlled bracket citations such as [Monos]. Citation metrics remain
# separate from plain brand-mention metrics.
PRODUCT_MENTION_ALIASES = {
    "Away": ["Away", "LifetimeCare"],
    "BR": ["Briggs & Riley", "Briggs and Riley", "Briggs", "Baseline Essential", "Baseline", "CX", "Outsider"],
    "Century": ["Hartmann", "Century Deluxe", "Century"],
    "July": ["July", "SilentMove", "Y-Strap"],
    "Rimowa": ["RIMOWA", "Rimowa", "Essential Lite", "Multiwheel"],
}

MENTION_CANONICAL = {}
for _canon, _aliases in PRODUCT_MENTION_ALIASES.items():
    for _alias in _aliases:
        MENTION_CANONICAL[_alias.lower()] = _canon

# Match longer aliases first so "Platinum Elite" is not split into smaller fragments.
MENTION_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(a) for aliases in PRODUCT_MENTION_ALIASES.values() for a in sorted(aliases, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

POSITIVE_CONTEXT_PATTERN = re.compile(
    r"\b(best|strong|recommend(?:ed)?|useful|durable|smooth|secure|clear|confidence|better|stand(?:s)? out|excellent|ideal|good|reliable|premium|spacious|lightweight|protect(?:s|ive|ion)?|organized|convenient|easy|fits?|warranty|trial|trusted|effortless|precise|room|capacity)\b",
    flags=re.IGNORECASE,
)
NEGATIVE_CONTEXT_PATTERN = re.compile(
    r"\b(caution|limitation|limited|not|may not|does not|doesn't|heavy|larger|weaker|less|risk|gate-check|gate check|check|expensive|unclear|missing|lacks?|trade[- ]?off|downside|fully expanded|may exceed|harder)\b",
    flags=re.IGNORECASE,
)

def answer_without_product_citations(text: str) -> str:
    return PRODUCT_PATTERN.sub(" ", text or "")

def canonical_mention(label: str) -> str:
    return MENTION_CANONICAL.get((label or "").lower(), label)

def brand_mention_sequence(text: str):
    clean_text = answer_without_product_citations(text)
    return [canonical_mention(m.group(1)) for m in MENTION_PATTERN.finditer(clean_text or "")]

def product_mentioned_in_text(text: str, product: str) -> bool:
    return product in set(brand_mention_sequence(text))

def mentioned_sentences_for_product(answer: str, product: str):
    sentences = []
    for sent in split_sentences(answer_without_product_citations(answer)):
        if product_mentioned_in_text(sent, product):
            sentences.append(sent)
    return sentences

def simple_brand_sentiment(context_text: str):
    # Lightweight dictionary proxy for answer-context tone. This is descriptive only,
    # not a substitute for a validated sentiment model.
    pos = len(POSITIVE_CONTEXT_PATTERN.findall(context_text or ""))
    neg = len(NEGATIVE_CONTEXT_PATTERN.findall(context_text or ""))
    denom = pos + neg
    score = (pos - neg) / denom if denom else 0.0
    return float(score), int(pos), int(neg)


def split_sentences(text: str):
    parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
    return [p.strip() for p in parts if p.strip()]

def cited_sentences_for_product(answer: str, product: str):
    sentences = []
    for sent in split_sentences(answer):
        if product in set(citation_sequence(sent)):
            sentences.append(sent)
    return sentences

def categories_in_text(text: str):
    found = []
    for cat, pat in FEATURE_PATTERNS.items():
        if re.search(pat, text or "", flags=re.IGNORECASE):
            found.append(cat)
    return sorted(set(found))

def product_feature_categories(answer: str, product: str):
    cats = set()
    for sent in cited_sentences_for_product(answer, product):
        cats.update(categories_in_text(sent))
    return sorted(cats)

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))

def simple_ngram_overlap(a: str, b: str, n: int = 2) -> float:
    def toks(x):
        return re.findall(r"\b[a-zA-Z0-9]+\b", (x or "").lower())
    ta, tb = toks(a), toks(b)
    if len(ta) < n or len(tb) < n:
        return 0.0
    nga = set(tuple(ta[i:i+n]) for i in range(len(ta)-n+1))
    ngb = set(tuple(tb[i:i+n]) for i in range(len(tb)-n+1))
    if not nga or not ngb:
        return 0.0
    return len(nga & ngb) / len(ngb)

def tfidf_similarity_to_product_text(product_text: str, cited_answer_text: str) -> float:
    if not SKLEARN_AVAILABLE:
        return 0.0
    if not product_text.strip() or not cited_answer_text.strip():
        return 0.0
    try:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        X = vec.fit_transform([product_text, cited_answer_text])
        return float(cosine_similarity(X[0:1], X[1:2])[0, 0])
    except Exception:
        return 0.0

def build_question_product_metrics(parsed_df: pd.DataFrame, mode: str) -> pd.DataFrame:
    rows = []

    for _, row in parsed_df.iterrows():
        source_file = row["source_file"]
        source_path = row.get("source_path", "")
        run_type = row["run_type"]
        focal_current_product = row.get("focal_current_product", None)
        qid = int(row["question_id"])
        answer = row["answer_text"]
        seq = citation_sequence(answer)
        total_product_ref_count = len(seq)
        mention_seq = brand_mention_sequence(answer)
        total_brand_mention_count = len(mention_seq)

        unique_order = []
        for p in seq:
            if p not in unique_order:
                unique_order.append(p)

        first_unique = unique_order[0] if unique_order else None

        unique_mention_order = []
        for p in mention_seq:
            if p not in unique_mention_order:
                unique_mention_order.append(p)
        first_mentioned_unique = unique_mention_order[0] if unique_mention_order else None

        for product in PRODUCTS:
            if mode == "baseline":
                version_status = "original_2023_2024"
            else:
                version_status = "current_2026" if product == focal_current_product else "original_2023_2024"

            ref_count = seq.count(product)
            cited = ref_count > 0
            first_idx = seq.index(product) + 1 if cited else np.nan
            first_score = 1 / first_idx if cited else 0.0
            citation_share_of_voice = ref_count / total_product_ref_count if total_product_ref_count else 0.0
            citation_position_proxy = first_idx if cited else N_PRODUCTS + 1
            citation_position_score = max((N_PRODUCTS + 1 - citation_position_proxy) / N_PRODUCTS, 0)

            brand_mention_count = mention_seq.count(product)
            brand_mentioned = brand_mention_count > 0
            first_mention_idx = mention_seq.index(product) + 1 if brand_mentioned else np.nan
            first_mention_score = 1 / first_mention_idx if brand_mentioned else 0.0
            mention_share_of_voice = brand_mention_count / total_brand_mention_count if total_brand_mention_count else 0.0
            mention_position_proxy = unique_mention_order.index(product) + 1 if product in unique_mention_order else N_PRODUCTS + 1
            mention_position_score = max((N_PRODUCTS + 1 - mention_position_proxy) / N_PRODUCTS, 0)
            average_mention_position_proxy = mention_position_proxy

            recommendation_rank_proxy = unique_order.index(product) + 1 if product in unique_order else N_PRODUCTS + 1
            rank_score_proxy = max((N_PRODUCTS + 1 - recommendation_rank_proxy) / N_PRODUCTS, 0)

            cited_sents = cited_sentences_for_product(answer, product)
            mention_sents = mentioned_sentences_for_product(answer, product)
            cited_answer_text = " ".join(cited_sents)
            mentioned_answer_text = " ".join(mention_sents)
            product_text = product_texts.get((version_status, product), "")

            feature_cats = product_feature_categories(answer, product)
            feature_coverage = len(feature_cats) / len(FEATURE_PATTERNS)
            tfidf_sim = tfidf_similarity_to_product_text(product_text, cited_answer_text) if cited else 0.0
            ngram_overlap = simple_ngram_overlap(product_text, cited_answer_text, n=2) if cited else 0.0
            brand_context_text = " ".join([cited_answer_text, mentioned_answer_text]).strip()
            brand_context_sentiment_score, brand_context_positive_hits, brand_context_negative_hits = simple_brand_sentiment(brand_context_text)

            ref_component = min(ref_count / 3, 1)
            first_component = first_score

            # Product Answer Influence Score, adapted for this benchmark.
            # This is an observational proxy, not a causal or attention-based metric.
            pais = (
                0.25 * ref_component
                + 0.20 * first_component
                + 0.20 * feature_coverage
                + 0.20 * tfidf_sim
                + 0.15 * ngram_overlap
            )

            rows.append({
                "comparison": COMPARISON_NAME,
                "mode": mode,
                "source_file": source_file,
                "source_path": source_path,
                "run_type": run_type,
                "focal_current_product": focal_current_product,
                "version_status_for_product": version_status,
                "prompt_set_start": row.get("prompt_set_start", np.nan),
                "prompt_set_end": row.get("prompt_set_end", np.nan),
                "prompt_set_label": row.get("prompt_set_label", ""),
                "question_id": qid,
                "product": product,
                "cited": int(cited),
                "ref_count": ref_count,
                "first_citation_index": first_idx,
                "first_citation_score": first_score,
                "citation_position_proxy": citation_position_proxy,
                "citation_position_score": citation_position_score,
                "citation_share_of_voice": citation_share_of_voice,
                "recommendation_rank_proxy": recommendation_rank_proxy,
                "rank_score_proxy": rank_score_proxy,
                "top1_cited_flag": int(product == first_unique),
                "brand_mentioned": int(brand_mentioned),
                "brand_mention_count": brand_mention_count,
                "first_mention_index": first_mention_idx,
                "first_mention_score": first_mention_score,
                "mention_share_of_voice": mention_share_of_voice,
                "mention_position_proxy": mention_position_proxy,
                "mention_position_score": mention_position_score,
                "average_mention_position_proxy": average_mention_position_proxy,
                "top1_mentioned_flag": int(product == first_mentioned_unique),
                "feature_categories": ";".join(feature_cats),
                "feature_count": len(feature_cats),
                "feature_coverage": feature_coverage,
                "tfidf_product_answer_similarity": tfidf_sim,
                "bigram_overlap_product_answer": ngram_overlap,
                "pais_product_answer_influence_score": pais,
                "brand_context_sentiment_score": brand_context_sentiment_score,
                "brand_context_positive_hits": brand_context_positive_hits,
                "brand_context_negative_hits": brand_context_negative_hits,
                "answer_word_count": word_count(answer),
                "citation_sequence": " > ".join(seq),
                "cited_answer_sentences": cited_answer_text,
                "mentioned_answer_sentences": mentioned_answer_text,
            })

    return pd.DataFrame(rows)

baseline_qp_metrics = build_question_product_metrics(baseline_answers, mode="baseline")
current_qp_metrics = build_question_product_metrics(current_answers, mode="one_product_current")

display(baseline_qp_metrics.head(15))
display(current_qp_metrics.head(15))


,comparison,mode,source_file,source_path,run_type,focal_current_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label,...,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,brand_context_sentiment_score,brand_context_positive_hits,brand_context_negative_hits,answer_word_count,citation_sequence,cited_answer_sentences,mentioned_answer_sentences
0,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.117537,0.275362,0.494636,0.714286,6,1,107,Away > Away > July,"[Away] It also says the suitcase is sized to fit in all overhead bins, is perfect for any quick trip, and meets carry-on requirements for most major airlines. [Away] July is a ...","I would recommend Away The Carry-On for a 3-5 day trip because its page explicitly labels it for 3-5 days, says it fits 5-7 outfits, and lists a 39.8 L capacity."
1,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,107,Away > Away > July,,
2,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,107,Away > Away > July,,
3,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.122061,0.000000,0.174412,0.333333,2,1,107,Away > Away > July,[July],"July is a strong alternative if you want slightly more listed capacity, because it offers 42 L, an ejectable battery, a hidden laundry bag, and Y-Strap compression, but its pag..."
4,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,107,Away > Away > July,,
5,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,108,July > July > BR,,
6,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.063532,1.000000,0.312706,1.000000,1,0,108,July > July > BR,[Briggs & Riley],"Briggs & Riley is the strongest softside durability runner-up because its page lists remarkably strong ballistic nylon that resists wear, impact-resistant corner guards, self-r..."
7,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,108,July > July > BR,,
8,01_2023_2024_original_vs_2026_current,baseline,baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_all_original,None,original_2023_2024,1,5,1-5,...,0.089485,0.163934,0.440733,1.000000,3,0,1

,comparison,mode,source_file,source_path,run_type,focal_current_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label,...,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,brand_context_sentiment_score,brand_context_positive_hits,brand_context_negative_hits,answer_word_count,citation_sequence,cited_answer_sentences,mentioned_answer_sentences
0,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,current_2026,1,5,1-5,...,0.104034,0.256410,0.541725,0.777778,8,1,123,Away > Away > July,"[Away] It also adds a compression system, three mesh pockets, one hanging pocket, a TSA-approved combination lock, smooth-gliding wheels, and a water-resistant laundry bag, so ...","I would recommend Away The Carry-On first for a 3-5 day trip because its page explicitly says it is perfect for 3-5 day trips, packs 5-7 outfits, fits overhead on most major ai..."
1,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,123,Away > Away > July,,
2,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,123,Away > Away > July,,
3,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,original_2023_2024,1,5,1-5,...,0.122061,0.000000,0.174412,0.600000,4,1,123,Away > Away > July,[July],"July is also a strong option if you want a little more stated capacity at 42 L plus an ejectable USB-C battery and lifetime warranty, but its page does not specifically describ..."
4,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,original_2023_2024,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,123,Away > Away > July,,
5,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,current_2026,1,5,1-5,...,0.102574,0.000000,0.170515,1.000000,1,0,119,July > BR > Away,[Away],"Away also looks durable because its page says the 100% polycarbonate shell is rigorously tested, and it describes handle, wheel, zipper, and drop testing."
6,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,original_2023_2024,1,5,1-5,...,0.032415,0.040000,0.248448,1.000000,3,0,119,July > BR > Away,"[Briggs & Riley] Away also looks durable because its page says the 100% polycarbonate shell is rigorously tested, and it describes handle, wheel, zipper, and drop testing.","Briggs & Riley is the best soft-sided durability alternative because it uses strong ballistic nylon, impact-resistant corner guards, self-repairing YKK zippers, and a guarantee..."
7,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users

## Match current runs to baseline question IDs

This is important for multi-prompt analysis.

If current files include Q1–5, Q6–10, and Q16–20, then the baseline comparison should use only those same question IDs.  
If Q11–15 GEO files are accidentally in the current folder, they are ignored in Comparison 01.


In [8]:
# =============================
# Matched current-vs-baseline question-product deltas
# =============================

METRIC_COLS = [
    "cited",
    "ref_count",
    "first_citation_score",
    "recommendation_rank_proxy",
    "rank_score_proxy",
    "top1_cited_flag",
    "feature_count",
    "feature_coverage",
    "tfidf_product_answer_similarity",
    "bigram_overlap_product_answer",
    "pais_product_answer_influence_score",
    "brand_mentioned",
    "brand_mention_count",
    "mention_share_of_voice",
    "citation_share_of_voice",
    "first_mention_score",
    "mention_position_proxy",
    "mention_position_score",
    "average_mention_position_proxy",
    "citation_position_proxy",
    "citation_position_score",
    "brand_context_sentiment_score",
    "brand_context_positive_hits",
    "brand_context_negative_hits",
]

current_focal_rows = current_qp_metrics[
    current_qp_metrics["product"] == current_qp_metrics["focal_current_product"]
].copy()

baseline_lookup_cols = [
    "product",
    "question_id",
    "source_file",
    "prompt_set_label",
] + METRIC_COLS

baseline_lookup = baseline_qp_metrics[baseline_lookup_cols].copy()

# If multiple baseline files contain the same product/question_id, average them.
# This supports repeated baseline runs while keeping the comparison stable.
agg_dict = {col: "mean" for col in METRIC_COLS}
agg_dict.update({
    "source_file": lambda x: ";".join(sorted(set(map(str, x)))),
    "prompt_set_label": lambda x: ";".join(sorted(set(map(str, x)))),
})
baseline_lookup = (
    baseline_lookup
    .groupby(["product", "question_id"], as_index=False)
    .agg(agg_dict)
)

matched = current_focal_rows.merge(
    baseline_lookup,
    on=["product", "question_id"],
    how="left",
    suffixes=("_current", "_baseline"),
)

matched["baseline_match_found"] = matched["source_file_baseline"].notna()

missing_baseline_matches = matched[~matched["baseline_match_found"]].copy()
current_question_ids = set(map(int, current_focal_rows["question_id"].unique()))
baseline_question_ids = set(map(int, baseline_lookup["question_id"].unique()))
matched_question_ids = set(map(int, matched[matched["baseline_match_found"]]["question_id"].unique()))
missing_baseline_question_ids = sorted(current_question_ids - baseline_question_ids)
missing_current_question_ids_vs_expected = sorted(set(EXPECTED_QUESTION_IDS) - current_question_ids)
missing_baseline_question_ids_vs_expected = sorted(set(EXPECTED_QUESTION_IDS) - baseline_question_ids)

matching_coverage_summary = pd.DataFrame([{
    "n_current_focal_rows": len(current_focal_rows),
    "n_matched_rows": int(matched["baseline_match_found"].sum()),
    "n_unmatched_current_rows": int((~matched["baseline_match_found"]).sum()),
    "current_question_ids": ",".join(map(str, sorted(current_question_ids))),
    "baseline_question_ids": ",".join(map(str, sorted(baseline_question_ids))),
    "matched_question_ids": ",".join(map(str, sorted(matched_question_ids))),
    "missing_baseline_question_ids_for_current": ",".join(map(str, missing_baseline_question_ids)),
    "missing_current_question_ids_vs_expected": ",".join(map(str, missing_current_question_ids_vs_expected)),
    "missing_baseline_question_ids_vs_expected": ",".join(map(str, missing_baseline_question_ids_vs_expected)),
}])

print("Matching coverage summary:")
display(matching_coverage_summary)

if not missing_baseline_matches.empty:
    print("WARNING: Some current focal rows do not have matching baseline question IDs.")
    display(
        missing_baseline_matches[
            ["source_file_current", "focal_current_product", "product", "question_id", "prompt_set_label_current"]
        ].sort_values(["product", "question_id", "source_file_current"])
    )

if missing_current_question_ids_vs_expected:
    print("NOTE: These expected question IDs are not present in included current-vs-2026 files:", missing_current_question_ids_vs_expected)
    print("If Q11-15 files are currently named with 'vsGEO', they are intentionally ignored in Comparison 01. Rename them to '*2023vs2026_11.txt' only if they are truly current-vs-2026 runs.")

if (not ALLOW_PARTIAL_MATCH) and missing_baseline_question_ids:
    raise ValueError(
        "Cannot compute a full multi-prompt current-vs-baseline delta because some current question IDs do not have matching all-original baseline answers. "
        f"Missing baseline question IDs for current files: {missing_baseline_question_ids}. "
        "Add all-original baseline answer files for those questions to the baseline folder, or set ALLOW_PARTIAL_MATCH = True if you intentionally want a partial matched analysis."
    )

matched_complete = matched[matched["baseline_match_found"]].copy()

for col in METRIC_COLS:
    matched_complete[f"delta_{col}"] = matched_complete[f"{col}_current"] - matched_complete[f"{col}_baseline"]

# More readable aliases for downstream tables.
matched_complete["delta_citation_rate"] = matched_complete["delta_cited"]
matched_complete["delta_mean_ref_count"] = matched_complete["delta_ref_count"]
matched_complete["rank_improvement_current_minus_baseline"] = (
    matched_complete["recommendation_rank_proxy_baseline"] - matched_complete["recommendation_rank_proxy_current"]
)
matched_complete["delta_top1_cited_share"] = matched_complete["delta_top1_cited_flag"]
matched_complete["delta_pais"] = matched_complete["delta_pais_product_answer_influence_score"]

print("Current focal rows:", len(current_focal_rows))
print("Matched rows used for delta:", len(matched_complete))
display(matched_complete.head(30))


Matching coverage summary:


,n_current_focal_rows,n_matched_rows,n_unmatched_current_rows,current_question_ids,baseline_question_ids,matched_question_ids,missing_baseline_question_ids_for_current,missing_current_question_ids_vs_expected,missing_baseline_question_ids_vs_expected
0,100,100,0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",,,


Current focal rows: 100
Matched rows used for delta: 100


,comparison,mode,source_file_current,source_path,run_type,focal_current_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label_current,...,delta_citation_position_proxy,delta_citation_position_score,delta_brand_context_sentiment_score,delta_brand_context_positive_hits,delta_brand_context_negative_hits,delta_citation_rate,delta_mean_ref_count,rank_improvement_current_minus_baseline,delta_top1_cited_share,delta_pais
0,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,current_2026,1,5,1-5,...,0.0,0.0,0.063492,2.0,0.0,0.0,0.0,0.0,0.0,0.047088
1,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,current_2026,1,5,1-5,...,-3.0,0.6,1.000000,1.0,0.0,1.0,1.0,3.0,0.0,0.170515
2,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,current_2026,1,5,1-5,...,2.0,-0.4,-1.000000,-3.0,0.0,-1.0,-1.0,-3.0,0.0,-0.137948
3,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,current_2026,1,5,1-5,...,1.0,-0.2,0.000000,2.0,0.0,0.0,0.0,-1.0,-1.0,-0.106400
4,01_2023_2024_original_vs_2026_current,one_product_current,away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,one_product_current,Away,current_2026,1,5,1-5,...,-1.0,0.2,0.171429,3.0,0.0,0.0,1.0,1.0,1.0,0.202224
5,01_2023_2024_original_vs_2026_current,one_product_current,away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_11.txt,one_product_current,Away,current_2026,11,15,11-15,...,0.0,0.0,0.000000,-4.0,0.0,0.0,0.0,0.0,0.0,0.018377
6,01_2023_2024_original_vs_2026_current,one_product_current,away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_11.txt,one_product_current,Away,current_2026,11,15,11-15,...,0.0,0.0,-0.133333,-1.0,0.0,0.0,0.0,0.0,0.0,0.001762
7,01_2023_2024_original_vs_2026_current,one_product_current,away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_11.txt,one_product_current,Away,current_2026,11,15,11-15,...,0.0,0.0,-0.833333,-8.0,-1.0,0.0,-1.0,0.0,0.0,-0.117450
8,01_2023_2024_original_vs_2026_current,one_product_current,away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_11.txt,one_product_current,Away,current_2026,11,15,11-15,...,1.0,-0.2,0.500000,0.0,-1.0,0.0,1.0,0.0,0.0,0.079681
9,01_2023_2024_original_vs_2026_current,one_product_current,away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_11.txt,one_product_current,Away,current_2026,11,15,11-15,...,0.0,0.0,0.333333,0.0,-2.0,0.0,-2.0,0.0,0.0,-0.206655


## Product summaries

The main product-level summary is now based on **matched current-vs-baseline rows**.

This prevents unfair comparison if some prompt sets exist for current files but not in the baseline files, or vice versa.


In [9]:
# =============================
# Product-level summaries from matched rows
# =============================

def summarize_side_from_matched(matched_df: pd.DataFrame, side: str, label: str) -> pd.DataFrame:
    suffix = "_current" if side == "current" else "_baseline"

    temp = pd.DataFrame({
        "product": matched_df["product"],
        "question_id": matched_df["question_id"],
        "source_file_current": matched_df["source_file_current"],
        "source_file_baseline": matched_df["source_file_baseline"],
        "prompt_set_label_current": matched_df["prompt_set_label_current"],
        "cited": matched_df[f"cited{suffix}"],
        "ref_count": matched_df[f"ref_count{suffix}"],
        "first_citation_score": matched_df[f"first_citation_score{suffix}"],
        "recommendation_rank_proxy": matched_df[f"recommendation_rank_proxy{suffix}"],
        "rank_score_proxy": matched_df[f"rank_score_proxy{suffix}"],
        "top1_cited_flag": matched_df[f"top1_cited_flag{suffix}"],
        "feature_count": matched_df[f"feature_count{suffix}"],
        "feature_coverage": matched_df[f"feature_coverage{suffix}"],
        "tfidf_product_answer_similarity": matched_df[f"tfidf_product_answer_similarity{suffix}"],
        "bigram_overlap_product_answer": matched_df[f"bigram_overlap_product_answer{suffix}"],
        "pais_product_answer_influence_score": matched_df[f"pais_product_answer_influence_score{suffix}"],
    })

    summary = (
        temp.groupby("product")
        .agg(
            n_question_contexts=("question_id", "count"),
            question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(x))))),
            n_current_source_files=("source_file_current", "nunique"),
            n_baseline_source_files=("source_file_baseline", "nunique"),
            prompt_sets=("prompt_set_label_current", lambda x: join_prompt_set_labels(x)),
            citation_rate=("cited", "mean"),
            mean_ref_count=("ref_count", "mean"),
            total_ref_count=("ref_count", "sum"),
            mean_first_citation_score=("first_citation_score", "mean"),
            mean_recommendation_rank_proxy=("recommendation_rank_proxy", "mean"),
            mean_rank_score_proxy=("rank_score_proxy", "mean"),
            top1_cited_share=("top1_cited_flag", "mean"),
            mean_feature_count=("feature_count", "mean"),
            mean_feature_coverage=("feature_coverage", "mean"),
            mean_tfidf_product_answer_similarity=("tfidf_product_answer_similarity", "mean"),
            mean_bigram_overlap_product_answer=("bigram_overlap_product_answer", "mean"),
            mean_pais_product_answer_influence_score=("pais_product_answer_influence_score", "mean"),
        )
        .reset_index()
    )
    summary.insert(0, "metric_source", label)
    return summary

baseline_product_summary_matched = summarize_side_from_matched(
    matched_complete,
    side="baseline",
    label="baseline_all_original_matched_questions",
)

current_product_summary_matched = summarize_side_from_matched(
    matched_complete,
    side="current",
    label="one_product_current_focal_matched_questions",
)

display(baseline_product_summary_matched)
display(current_product_summary_matched)


,metric_source,product,n_question_contexts,question_ids,n_current_source_files,n_baseline_source_files,prompt_sets,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,baseline_all_original_matched_questions,Away,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.35,27.0,0.583333,2.40,0.72,0.45,5.30,0.278947,0.046991,0.059469,0.303275
1,baseline_all_original_matched_questions,BR,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.85,1.45,29.0,0.386786,3.05,0.59,0.20,5.15,0.271053,0.055961,0.173692,0.281314
2,baseline_all_original_matched_questions,Century,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.75,1.15,23.0,0.227879,4.20,0.36,0.05,4.15,0.218421,0.041144,0.044793,0.200041
3,baseline_all_original_matched_questions,July,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.55,31.0,0.445833,2.65,0.67,0.20,4.35,0.228947,0.073160,0.051775,0.286521
4,baseline_all_original_matched_questions,Rimowa,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.75,1.05,21.0,0.232976,4.25,0.35,0.10,3.40,0.178947,0.037249,0.011805,0.179105


,metric_source,product,n_question_contexts,question_ids,n_current_source_files,n_baseline_source_files,prompt_sets,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,one_product_current_focal_matched_questions,Away,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.85,1.25,25,0.622500,2.35,0.73,0.50,4.95,0.260526,0.059769,0.070781,0.303343
1,one_product_current_focal_matched_questions,BR,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.35,27,0.496310,2.75,0.65,0.30,4.65,0.244737,0.064089,0.175402,0.299837
2,one_product_current_focal_matched_questions,Century,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.80,1.35,27,0.257202,4.05,0.39,0.10,3.35,0.176316,0.053101,0.035150,0.215096
3,one_product_current_focal_matched_questions,July,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.20,24,0.530833,2.30,0.74,0.30,4.05,0.213158,0.062739,0.039088,0.267209
4,one_product_current_focal_matched_questions,Rimowa,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.65,0.85,17,0.164921,4.85,0.23,0.05,2.75,0.144737,0.038603,0.011172,0.142161


## Current-vs-baseline delta by product

This is the main output.

Positive delta means the product became more visible or more prominent when its own page was replaced by the 2026 current version, compared with the matched all-original baseline.


In [10]:
# =============================
# Current - baseline product-level delta
# =============================

def make_current_vs_baseline_delta_from_matched(matched_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for product, g in matched_df.groupby("product"):
        rows.append({
            "comparison": COMPARISON_NAME,
            "product": product,
            "matched_n_question_contexts": int(len(g)),
            "question_ids": ",".join(map(str, sorted(g["question_id"].unique()))),
            "current_source_files": ";".join(sorted(g["source_file_current"].unique())),
            "baseline_source_files": ";".join(sorted(g["source_file_baseline"].unique())),
            "prompt_sets": join_prompt_set_labels(g["prompt_set_label_current"]),

            "baseline_citation_rate": g["cited_baseline"].mean(),
            "current_citation_rate": g["cited_current"].mean(),
            "delta_citation_rate": g["delta_cited"].mean(),

            "baseline_mean_ref_count": g["ref_count_baseline"].mean(),
            "current_mean_ref_count": g["ref_count_current"].mean(),
            "delta_mean_ref_count": g["delta_ref_count"].mean(),

            "baseline_first_citation_score": g["first_citation_score_baseline"].mean(),
            "current_first_citation_score": g["first_citation_score_current"].mean(),
            "delta_first_citation_score": g["delta_first_citation_score"].mean(),

            "baseline_mean_recommendation_rank_proxy": g["recommendation_rank_proxy_baseline"].mean(),
            "current_mean_recommendation_rank_proxy": g["recommendation_rank_proxy_current"].mean(),
            "rank_improvement_current_minus_baseline": (
                g["recommendation_rank_proxy_baseline"].mean() - g["recommendation_rank_proxy_current"].mean()
            ),

            "baseline_rank_score_proxy": g["rank_score_proxy_baseline"].mean(),
            "current_rank_score_proxy": g["rank_score_proxy_current"].mean(),
            "delta_rank_score_proxy": g["delta_rank_score_proxy"].mean(),

            "baseline_top1_cited_share": g["top1_cited_flag_baseline"].mean(),
            "current_top1_cited_share": g["top1_cited_flag_current"].mean(),
            "delta_top1_cited_share": g["delta_top1_cited_flag"].mean(),

            "baseline_feature_coverage": g["feature_coverage_baseline"].mean(),
            "current_feature_coverage": g["feature_coverage_current"].mean(),
            "delta_feature_coverage": g["delta_feature_coverage"].mean(),

            "baseline_pais": g["pais_product_answer_influence_score_baseline"].mean(),
            "current_pais": g["pais_product_answer_influence_score_current"].mean(),
            "delta_pais": g["delta_pais_product_answer_influence_score"].mean(),
        })

    delta = pd.DataFrame(rows)

    # Composite descriptive score.
    # This is not a causal estimate; it summarizes observed answer-behavior movement.
    delta["current_page_advantage_score"] = (
        0.25 * delta["delta_rank_score_proxy"]
        + 0.20 * delta["delta_citation_rate"]
        + 0.20 * delta["delta_first_citation_score"]
        + 0.15 * delta["delta_top1_cited_share"]
        + 0.10 * delta["delta_feature_coverage"]
        + 0.10 * delta["delta_pais"]
    )

    delta["direction_label"] = np.select(
        [
            delta["current_page_advantage_score"] > 0.05,
            delta["current_page_advantage_score"] < -0.05,
        ],
        [
            "current_2026_stronger",
            "baseline_2023_2024_stronger",
        ],
        default="similar_or_small_change",
    )

    return delta.sort_values("current_page_advantage_score", ascending=False)

current_vs_baseline_delta_by_product = make_current_vs_baseline_delta_from_matched(matched_complete)

display(current_vs_baseline_delta_by_product)


,comparison,product,matched_n_question_contexts,question_ids,current_source_files,baseline_source_files,prompt_sets,baseline_citation_rate,current_citation_rate,delta_citation_rate,...,current_top1_cited_share,delta_top1_cited_share,baseline_feature_coverage,current_feature_coverage,delta_feature_coverage,baseline_pais,current_pais,delta_pais,current_page_advantage_score,direction_label
1,01_2023_2024_original_vs_2026_current,BR,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",br_1.txt;br_11.txt;br_16.txt;br_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.85,0.90,0.05,...,0.30,0.10,0.271053,0.244737,-0.026316,0.281314,0.299837,0.018524,0.061126,current_2026_stronger
3,01_2023_2024_original_vs_2026_current,July,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",july_1.txt;july_11.txt;july_16.txt;july_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.90,0.90,0.00,...,0.30,0.10,0.228947,0.213158,-0.015789,0.286521,0.267209,-0.019312,0.045990,similar_or_small_change
2,01_2023_2024_original_vs_2026_current,Century,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",century_1.txt;century_11.txt;century_16.txt;century_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.75,0.80,0.05,...,0.10,0.05,0.218421,0.176316,-0.042105,0.200041,0.215096,0.015056,0.028160,similar_or_small_change
0,01_2023_2024_original_vs_2026_current,Away,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",away_1.txt;away_11.txt;away_16.txt;away_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.90,0.85,-0.05,...,0.50,0.05,0.278947,0.260526,-0.018421,0.303275,0.303343,0.000068,0.005998,similar_or_small_change
4,01_2023_2024_original_vs_2026_current,Rimowa,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",rimowa_1.txt;rimowa_11.txt;rimowa_16.txt;rimowa_6.txt,baseline_1.txt;baseline_11.txt;baseline_16.txt;baseline_6.txt,1-5;6-10;11-15;16-20,0.75,0.65,-0.10,...,0.05,-0.05,0.178947,0.144737,-0.034211,0.179105,0.142161,-0.036944,-0.078227,baseline_2023_2024_stronger


## Overall summary


In [11]:
overall_current_vs_baseline_summary = pd.DataFrame([{
    "comparison": COMPARISON_NAME,
    "purpose": "Check how much the actual product page changed over time",
    "interpretation": "General website/text change",
    "baseline_dir": str(BASELINE_DIR),
    "comparison_dir": str(COMPARISON_DIR),
    "n_baseline_files": len(BASELINE_FILES),
    "n_included_current_files": len(TXT_FILES),
    "n_ignored_txt_files": len(IGNORED_TXT_FILES),
    "included_question_ids": ",".join(map(str, sorted(matched_complete["question_id"].unique()))),
    "baseline_question_ids": ",".join(map(str, sorted(baseline_question_ids))),
    "current_question_ids": ",".join(map(str, sorted(current_question_ids))),
    "missing_baseline_question_ids_for_current": ",".join(map(str, missing_baseline_question_ids)),
    "missing_current_question_ids_vs_expected": ",".join(map(str, missing_current_question_ids_vs_expected)),
    "n_products": len(current_vs_baseline_delta_by_product),
    "mean_delta_citation_rate": current_vs_baseline_delta_by_product["delta_citation_rate"].mean(),
    "mean_delta_rank_score_proxy": current_vs_baseline_delta_by_product["delta_rank_score_proxy"].mean(),
    "mean_rank_improvement_current_minus_baseline": current_vs_baseline_delta_by_product["rank_improvement_current_minus_baseline"].mean(),
    "mean_delta_first_citation_score": current_vs_baseline_delta_by_product["delta_first_citation_score"].mean(),
    "mean_delta_top1_cited_share": current_vs_baseline_delta_by_product["delta_top1_cited_share"].mean(),
    "mean_delta_feature_coverage": current_vs_baseline_delta_by_product["delta_feature_coverage"].mean(),
    "mean_delta_pais": current_vs_baseline_delta_by_product["delta_pais"].mean(),
    "mean_current_page_advantage_score": current_vs_baseline_delta_by_product["current_page_advantage_score"].mean(),
    "n_current_2026_stronger": int((current_vs_baseline_delta_by_product["direction_label"] == "current_2026_stronger").sum()),
    "n_baseline_2023_2024_stronger": int((current_vs_baseline_delta_by_product["direction_label"] == "baseline_2023_2024_stronger").sum()),
    "n_similar_or_small_change": int((current_vs_baseline_delta_by_product["direction_label"] == "similar_or_small_change").sum()),
}])

display(overall_current_vs_baseline_summary)


,comparison,purpose,interpretation,baseline_dir,comparison_dir,n_baseline_files,n_included_current_files,n_ignored_txt_files,included_question_ids,baseline_question_ids,...,mean_delta_rank_score_proxy,mean_rank_improvement_current_minus_baseline,mean_delta_first_citation_score,mean_delta_top1_cited_share,mean_delta_feature_coverage,mean_delta_pais,mean_current_page_advantage_score,n_current_2026_stronger,n_baseline_2023_2024_stronger,n_similar_or_small_change
0,01_2023_2024_original_vs_2026_current,Check how much the actual product page changed over time,General website/text change,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current,4,20,0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",...,0.01,0.05,0.038992,0.05,-0.027368,-0.004522,0.012609,1,1,3


## Provider-style answer visibility metrics

In [12]:

# =============================
# Provider-style answer visibility metrics
# =============================
# These metrics translate the product-label answer files into commercial AI-visibility language.
# source/domain share is intentionally not computed here because the prompt uses controlled
# product-label citations such as [Monos], not external URL/domain citations.

PROVIDER_STYLE_METRIC_NOTE = (
    "Provider-style answer metrics added: brand_mention_rate, mention_share_of_voice, "
    "citation_share_of_voice, average_mention_position, average_citation_position, "
    "first_citation_score, sentiment context proxy, and comparison-level visibility delta. "
    "source/domain share is not available in this controlled product-label citation setting."
)

def _safe_mean(series):
    return pd.to_numeric(series, errors="coerce").mean()

def _provider_delta_rows(matched_df, left_suffix, right_suffix, left_name, right_name, improvement_label):
    rows = []
    for product, g in matched_df.groupby("product"):
        left_mention_pos = _safe_mean(g[f"average_mention_position_proxy{left_suffix}"])
        right_mention_pos = _safe_mean(g[f"average_mention_position_proxy{right_suffix}"])
        left_citation_pos = _safe_mean(g[f"citation_position_proxy{left_suffix}"])
        right_citation_pos = _safe_mean(g[f"citation_position_proxy{right_suffix}"])

        rows.append({
            "product": product,
            f"{left_name}_brand_mention_rate": _safe_mean(g[f"brand_mentioned{left_suffix}"]),
            f"{right_name}_brand_mention_rate": _safe_mean(g[f"brand_mentioned{right_suffix}"]),
            "delta_brand_mention_rate": _safe_mean(g[f"brand_mentioned{right_suffix}"] - g[f"brand_mentioned{left_suffix}"]),

            f"{left_name}_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{left_suffix}"]),
            f"{right_name}_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{right_suffix}"]),
            "delta_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{right_suffix}"] - g[f"mention_share_of_voice{left_suffix}"]),

            f"{left_name}_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{left_suffix}"]),
            f"{right_name}_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{right_suffix}"]),
            "delta_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{right_suffix}"] - g[f"citation_share_of_voice{left_suffix}"]),

            f"{left_name}_average_mention_position": left_mention_pos,
            f"{right_name}_average_mention_position": right_mention_pos,
            f"mention_position_improvement_{improvement_label}": left_mention_pos - right_mention_pos,

            f"{left_name}_average_citation_position": left_citation_pos,
            f"{right_name}_average_citation_position": right_citation_pos,
            f"citation_position_improvement_{improvement_label}": left_citation_pos - right_citation_pos,

            f"{left_name}_first_mention_score": _safe_mean(g[f"first_mention_score{left_suffix}"]),
            f"{right_name}_first_mention_score": _safe_mean(g[f"first_mention_score{right_suffix}"]),
            "delta_first_mention_score": _safe_mean(g[f"first_mention_score{right_suffix}"] - g[f"first_mention_score{left_suffix}"]),

            f"{left_name}_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{left_suffix}"]),
            f"{right_name}_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{right_suffix}"]),
            "delta_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{right_suffix}"] - g[f"brand_context_sentiment_score{left_suffix}"]),

            "source_domain_share_status": "not_available_controlled_product_label_citations_only",
        })
    out = pd.DataFrame(rows)
    out["provider_style_visibility_delta"] = (
        0.20 * out["delta_brand_mention_rate"]
        + 0.25 * out["delta_mention_share_of_voice"]
        + 0.25 * out["delta_citation_share_of_voice"]
        + 0.15 * (out[f"mention_position_improvement_{improvement_label}"] / N_PRODUCTS)
        + 0.15 * (out[f"citation_position_improvement_{improvement_label}"] / N_PRODUCTS)
    )
    return out

provider_answer_metric_delta_by_product = _provider_delta_rows(
    matched_complete,
    left_suffix="_baseline",
    right_suffix="_current",
    left_name="baseline",
    right_name="current",
    improvement_label="current_minus_baseline",
)
current_vs_baseline_delta_by_product = current_vs_baseline_delta_by_product.merge(
    provider_answer_metric_delta_by_product,
    on="product",
    how="left",
)
current_vs_baseline_delta_by_product["historical_visibility_delta"] = current_vs_baseline_delta_by_product["current_page_advantage_score"]

# Update overall summary so the saved overall CSV includes the provider-style layer.
overall_current_vs_baseline_summary["provider_style_metric_note"] = PROVIDER_STYLE_METRIC_NOTE
for _col in [
    "delta_brand_mention_rate",
    "delta_mention_share_of_voice",
    "delta_citation_share_of_voice",
    "provider_style_visibility_delta",
    "historical_visibility_delta",
    "delta_brand_context_sentiment_score",
]:
    overall_current_vs_baseline_summary[f"mean_{_col}"] = current_vs_baseline_delta_by_product[_col].mean()

display(current_vs_baseline_delta_by_product[[
    "product",
    "delta_brand_mention_rate",
    "delta_mention_share_of_voice",
    "delta_citation_share_of_voice",
    "mention_position_improvement_current_minus_baseline",
    "citation_position_improvement_current_minus_baseline",
    "delta_brand_context_sentiment_score",
    "provider_style_visibility_delta",
    "historical_visibility_delta",
]])


,product,delta_brand_mention_rate,delta_mention_share_of_voice,delta_citation_share_of_voice,mention_position_improvement_current_minus_baseline,citation_position_improvement_current_minus_baseline,delta_brand_context_sentiment_score,provider_style_visibility_delta,historical_visibility_delta
0,BR,0.05,0.008706,0.020433,0.30,0.65,-0.002020,0.045785,0.061126
1,July,0.00,-0.015176,-0.015482,0.35,0.50,0.254729,0.017836,0.045990
2,Century,0.00,0.023889,0.019739,0.00,0.15,-0.128127,0.015407,0.028160
3,Away,0.00,0.010463,0.000371,0.25,0.20,0.041501,0.016209,0.005998
4,Rimowa,-0.10,-0.027742,-0.030813,-0.60,-0.70,0.103690,-0.073639,-0.078227


## Save CSV outputs


In [13]:
file_manifest_comparison01.to_csv(OUTPUT_DIR / "file_manifest_comparison01.csv", index=False, encoding="utf-8-sig")
baseline_answers.to_csv(OUTPUT_DIR / "baseline_parsed_answers.csv", index=False, encoding="utf-8-sig")
current_answers.to_csv(OUTPUT_DIR / "current_parsed_answers_by_file.csv", index=False, encoding="utf-8-sig")
coverage_by_file.to_csv(OUTPUT_DIR / "question_coverage_by_file.csv", index=False, encoding="utf-8-sig")
question_coverage_summary.to_csv(OUTPUT_DIR / "question_coverage_summary.csv", index=False, encoding="utf-8-sig")
matching_coverage_summary.to_csv(OUTPUT_DIR / "matching_coverage_summary.csv", index=False, encoding="utf-8-sig")
missing_baseline_matches.to_csv(OUTPUT_DIR / "missing_baseline_matches.csv", index=False, encoding="utf-8-sig")
baseline_qp_metrics.to_csv(OUTPUT_DIR / "baseline_question_product_metrics.csv", index=False, encoding="utf-8-sig")
current_qp_metrics.to_csv(OUTPUT_DIR / "current_question_product_metrics.csv", index=False, encoding="utf-8-sig")
matched_complete.to_csv(OUTPUT_DIR / "matched_current_baseline_question_product_delta.csv", index=False, encoding="utf-8-sig")
baseline_product_summary_matched.to_csv(OUTPUT_DIR / "baseline_product_summary_matched.csv", index=False, encoding="utf-8-sig")
current_product_summary_matched.to_csv(OUTPUT_DIR / "current_product_summary_matched.csv", index=False, encoding="utf-8-sig")
current_vs_baseline_delta_by_product.to_csv(OUTPUT_DIR / "current_vs_baseline_delta_by_product.csv", index=False, encoding="utf-8-sig")
provider_answer_metric_delta_by_product.to_csv(OUTPUT_DIR / "provider_answer_metric_delta_by_product.csv", index=False, encoding="utf-8-sig")
overall_current_vs_baseline_summary.to_csv(OUTPUT_DIR / "overall_current_vs_baseline_summary.csv", index=False, encoding="utf-8-sig")
product_text_load_status.to_csv(OUTPUT_DIR / "product_text_load_status.csv", index=False, encoding="utf-8-sig")

print("Saved CSV files to:")
print(OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)


Saved CSV files to:
C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\tables\01_2023_2024_original_vs_2026_current_baseline_based
- baseline_parsed_answers.csv
- baseline_product_summary_matched.csv
- baseline_question_product_metrics.csv
- current_parsed_answers_by_file.csv
- current_product_summary_matched.csv
- current_question_product_metrics.csv
- current_vs_baseline_delta_by_product.csv
- file_manifest_comparison01.csv
- matched_current_baseline_question_product_delta.csv
- matching_coverage_summary.csv
- missing_baseline_matches.csv
- overall_current_vs_baseline_summary.csv
- product_text_load_status.csv
- provider_answer_metric_delta_by_product.csv
- question_coverage_by_file.csv
- question_coverage_summary.csv


## Quick interpretation table

Use this table first when writing the result.


In [14]:

cols = [
    "product",
    "direction_label",
    "current_page_advantage_score",
    "provider_style_visibility_delta",
    "historical_visibility_delta",
    "delta_rank_score_proxy",
    "delta_citation_rate",
    "delta_brand_mention_rate",
    "delta_citation_share_of_voice",
    "delta_mention_share_of_voice",
    "mention_position_improvement_current_minus_baseline",
    "citation_position_improvement_current_minus_baseline",
    "delta_first_citation_score",
    "delta_top1_cited_share",
    "delta_feature_coverage",
    "delta_pais",
    "matched_n_question_contexts",
    "question_ids",
    "prompt_sets",
]
cols = [c for c in cols if c in current_vs_baseline_delta_by_product.columns]
print("Quick check: provider-style answer metrics are now included alongside the original advantage metrics.")
print("source/domain share is not computed because these runs use controlled product-label citations, not external URLs/domains.")
display(current_vs_baseline_delta_by_product[cols])


Quick check: provider-style answer metrics are now included alongside the original advantage metrics.
source/domain share is not computed because these runs use controlled product-label citations, not external URLs/domains.


,product,direction_label,current_page_advantage_score,provider_style_visibility_delta,historical_visibility_delta,delta_rank_score_proxy,delta_citation_rate,delta_brand_mention_rate,delta_citation_share_of_voice,delta_mention_share_of_voice,mention_position_improvement_current_minus_baseline,citation_position_improvement_current_minus_baseline,delta_first_citation_score,delta_top1_cited_share,delta_feature_coverage,delta_pais,matched_n_question_contexts,question_ids,prompt_sets
0,BR,current_2026_stronger,0.061126,0.045785,0.061126,0.06,0.05,0.05,0.020433,0.008706,0.30,0.65,0.109524,0.10,-0.026316,0.018524,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
1,July,similar_or_small_change,0.045990,0.017836,0.045990,0.07,0.00,0.00,-0.015482,-0.015176,0.35,0.50,0.085000,0.10,-0.015789,-0.019312,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
2,Century,similar_or_small_change,0.028160,0.015407,0.028160,0.03,0.05,0.00,0.019739,0.023889,0.00,0.15,0.029324,0.05,-0.042105,0.015056,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
3,Away,similar_or_small_change,0.005998,0.016209,0.005998,0.01,-0.05,0.00,0.000371,0.010463,0.25,0.20,0.039167,0.05,-0.018421,0.000068,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
4,Rimowa,baseline_2023_2024_stronger,-0.078227,-0.073639,-0.078227,-0.12,-0.10,-0.10,-0.030813,-0.027742,-0.60,-0.70,-0.068056,-0.05,-0.034211,-0.036944,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20


## File-check helper

Use this cell if you want to quickly check whether filenames were classified correctly.

- `comparison01_current_included` = used in this notebook.
- `comparison01_ignored` = found but ignored, usually because it is a GEO file or not a current-replacement file.


In [15]:
display(file_manifest_comparison01.sort_values(["included", "file_role", "filename"], ascending=[False, True, True]))


,file_role,path,filename,detected_product,included,reason
0,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_1.txt,baseline_1.txt,,True,baseline answer file
1,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_11.txt,baseline_11.txt,,True,baseline answer file
2,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_16.txt,baseline_16.txt,,True,baseline answer file
3,baseline_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline\baseline_6.txt,baseline_6.txt,,True,baseline answer file
4,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_1.txt,away_1.txt,Away,True,current replacement file
5,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_11.txt,away_11.txt,Away,True,current replacement file
6,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_16.txt,away_16.txt,Away,True,current replacement file
7,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\away_6.txt,away_6.txt,Away,True,current replacement file
8,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\br_1.txt,br_1.txt,BR,True,current replacement file
9,comparison01_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\01_2023_2024_original_vs_2026_current\br_11.txt,br_11.txt,BR,True,current replacement file
